In [20]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
from collections import Counter

In [21]:
# Initialisation de la session Spark
spark = SparkSession \
    .builder \
    .appName("TradeCorp ETL") \
    .getOrCreate()

print(spark.version);



# Chemin relatif vers les CSV
filepath = "../data/tmp/"

# Création des DataFrames
print(f"Import des DataFrames")
df_categories = spark.read.parquet(filepath + "categories")
df_customers = spark.read.parquet(filepath + "clients")
df_employees = spark.read.parquet(filepath + "employees")
df_orders_details = spark.read.parquet(filepath + "details_commandes")
df_orders = spark.read.parquet(filepath + "commandes")
df_products = spark.read.parquet(filepath + "produits")
df_shippers = spark.read.parquet(filepath + "transporteurs")
df_suppliers = spark.read.parquet(filepath + "fournisseurs")

# Dictionnaire de tout les DataFrames
df_collection = {"categories" : df_categories, 
                 "clients" : df_customers, 
                 "employees" : df_employees, 
                 "details_commandes" : df_orders_details, 
                 "commandes" : df_orders, 
                 "produits" : df_products, 
                 "transporteurs" : df_shippers, 
                 "fournisseurs": df_suppliers}

# Vérification des DataFrames
for name, df in df_collection.items():
    print(f"Vérification du dataframe {name}")
    print(f"Nombre de lignes : {df.count()}")
    df.show(5)
    df.printSchema()
    print("\n")




4.2.0
Import des DataFrames
Vérification du dataframe categories
Nombre de lignes : 8
+-----------+--------------+--------------------+-------+
|category_id| category_name|         description|picture|
+-----------+--------------+--------------------+-------+
|          1|     Beverages|Soft drinks, coff...|   NULL|
|          2|    Condiments|Sweet and savory ...|   NULL|
|          3|   Confections|Desserts, candies...|   NULL|
|          4|Dairy Products|             Cheeses|   NULL|
|          5|Grains/Cereals|Breads, crackers,...|   NULL|
+-----------+--------------+--------------------+-------+
only showing top 5 rows
root
 |-- category_id: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- picture: string (nullable = true)



Vérification du dataframe clients
Nombre de lignes : 91
+-----------+--------------------+------------------+--------------------+--------------------+-----------+------+-----------+-------

# Q21 — Jointure orders + customers
Joindre df_orders et df_customers sur customer_id. Garder uniquement : order_id, company_name, country,
order_date, freight.

In [22]:
df_orders_by_company = df_orders \
    .join(df_customers, on="customer_id", how="inner") \
    .select("order_id", "company_name", "country", "order_date", "freight")

df_orders_by_company.show(5)

+--------+--------------------+-------+----------+-------+
|order_id|        company_name|country|order_date|freight|
+--------+--------------------+-------+----------+-------+
|   10400|  Eastern Connection|     UK|1997-01-01|  83.93|
|   10401|Rattlesnake Canyo...|    USA|1997-01-01|  12.51|
|   10402|        Ernst Handel|AUSTRIA|1997-01-02|  67.88|
|   10403|        Ernst Handel|AUSTRIA|1997-01-03|  73.79|
|   10404|Magazzini Aliment...|  ITALY|1997-01-03| 155.97|
+--------+--------------------+-------+----------+-------+
only showing top 5 rows


# Q22 — Jointure order_details + products
Joindre df_order_details et df_products sur product_id. Ajouter les colonnes product_name, category_id,
unit_price depuis products.

In [23]:
df_orders_details_with_products = df_orders_details \
    .join(df_products, on="product_id") \
    .select(df_orders_details["*"], df_products["product_name"], df_products["category_id"], df_products["unit_price"])

df_orders_details_with_products.show(5)

+--------+----------+-------------+--------+--------+----------+--------------------+-----------+----------+
|order_id|product_id|prix_unitaire|quantite|discount|sous_total|        product_name|category_id|unit_price|
+--------+----------+-------------+--------+--------+----------+--------------------+-----------+----------+
|   10248|        11|         14.0|      12|     0.0|     168.0|      Queso Cabrales|          4|      21.0|
|   10248|        72|         34.8|       5|     0.0|     174.0|Mozzarella di Gio...|          4|      34.8|
|   10249|        14|         18.6|       9|     0.0|     167.4|                Tofu|          7|     23.25|
|   10249|        51|         42.4|      40|     0.0|    1696.0|Manjimup Dried Ap...|          7|      53.0|
|   10250|        41|          7.7|      10|     0.0|      77.0|Jack's New Englan...|          8|      9.65|
+--------+----------+-------------+--------+--------+----------+--------------------+-----------+----------+
only showing top 5 

# Q23 — Jointure products + categories
Joindre df_products et df_categories sur category_id pour enrichir chaque produit avec category_name et
description.

In [24]:
df_products_with_categories = df_products \
    .join(df_categories, on="category_id") \
    .select(df_products["*"], "category_name", "description")

df_products_with_categories.show(5)

+----------+--------------------+-----------+-----------+-------------------+----------+--------------+--------------+-------------+------------+--------+-------------+--------------------+
|product_id|        product_name|supplier_id|category_id|  quantity_per_unit|unit_price|units_in_stock|units_on_order|reorder_level|discontinued|en_stock|category_name|         description|
+----------+--------------------+-----------+-----------+-------------------+----------+--------------+--------------+-------------+------------+--------+-------------+--------------------+
|         3|       Aniseed Syrup|          1|          2|12 - 550 ml bottles|      10.0|            13|            70|           25|           0|    true|   Condiments|Sweet and savory ...|
|         4|Chef Anton's Caju...|          2|          2|     48 - 6 oz jars|      22.0|            53|             0|            0|           0|    true|   Condiments|Sweet and savory ...|
|         6|Grandma's Boysenb...|          3|     

# Q24 — DataFrame enrichi complet
A - Réaliser une première jointure complète (order_details, orders, customers, products enrichi avec
categories, employees, shippers) sans renommer aucune colonne. Lister ensuite les colonnes qui apparaissent
en double grâce à Counter.

In [25]:
df_megajoin = df_orders_details \
    .join(df_orders, on="order_id") \
    .join(df_customers, on="customer_id") \
    .join(df_products_with_categories, on="product_id") \
    .join(df_employees, on="employee_id") \
    .join(df_shippers, on="shipper_id")

df_megajoin.show(5)

for name, count in Counter(df_megajoin.columns).items():
    if count > 1:
        print(f"Colonne en double : {name} : {count}")

+----------+-----------+----------+-----------+--------+-------------+--------+--------+----------+----------+-------------+------------+-------+--------------------+---------------+-----------+-----------+----------------+------------+----------+--------------------+------------+--------------------+---------------+-----------+------+-----------+-------+--------------+--------------+--------------------+-----------+-----------+------------------+----------+--------------+--------------+-------------+------------+--------+--------------+--------------------+----------+---------+--------------------+----------+-------+-------+-------------+----------------+--------------+
|shipper_id|employee_id|product_id|customer_id|order_id|prix_unitaire|quantite|discount|sous_total|order_date|required_date|shipped_date|freight|           ship_name|   ship_address|  ship_city|ship_region|ship_postal_code|ship_country|is_shipped|        company_name|contact_name|       contact_title|        address|  

B - Pour chaque colonne identifiée en Q24a, la renommer dans sa table d'origine avant de refaire la jointure,
en la préfixant selon la table (customer_country, employee_country, shipper_name...). Reconstruire ensuite
df_orders_enriched avec ces tables renommées, puis vérifier qu'il ne reste plus aucun doublon.

In [27]:
col_renamed_customers = {"company_name": "customer_company_name", "city" : "customer_city", "country" : "customer_country", "phone" : "customer_phone"}
col_renamed_shippers = {"company_name": "shipper_company_name", "phone" : "shipper_phone"}
col_renamed_employees = {"city" : "employee_city", "country" : "employee_country"}

df_customers = df_customers.withColumnsRenamed(col_renamed_customers)
df_shippers = df_shippers.withColumnsRenamed(col_renamed_shippers)
df_employees = df_employees.withColumnsRenamed(col_renamed_employees)

df_orders_enriched = df_orders_details \
    .join(df_orders, on="order_id") \
    .join(df_customers, on="customer_id") \
    .join(df_products_with_categories, on="product_id") \
    .join(df_employees, on="employee_id") \
    .join(df_shippers, on="shipper_id")

df_orders_enriched.show(5)

for name, count in Counter(df_orders_enriched.columns).items():
    if count > 1:
        print(f"Colonne en double : {name} : {count}")

+----------+-----------+----------+-----------+--------+-------------+--------+--------+----------+----------+-------------+------------+-------+--------------------+---------------+-----------+-----------+----------------+------------+----------+---------------------+------------+--------------------+---------------+-------------+------+-----------+----------------+--------------+--------------+--------------------+-----------+-----------+------------------+----------+--------------+--------------+-------------+------------+--------+--------------+--------------------+----------+---------+--------------------+----------+-------------+----------------+-------------+--------------------+--------------+
|shipper_id|employee_id|product_id|customer_id|order_id|prix_unitaire|quantite|discount|sous_total|order_date|required_date|shipped_date|freight|           ship_name|   ship_address|  ship_city|ship_region|ship_postal_code|ship_country|is_shipped|customer_company_name|contact_name|       

# Q25 — CA par client
Calculer le chiffre d'affaires total par client (company_name) depuis df_orders_enriched. Trier par CA
décroissant. Afficher le top 10.

In [ ]:
df_ca_by_company = df_orders_enriched \
        .groupby("customer_company_name") \
        .sum("sous_total") \
        .orderBy("sum(sous_total)", ascending=False)

df_ca_by_company.show(10)


+---------------------+------------------+
|customer_company_name|   sum(sous_total)|
+---------------------+------------------+
|           QUICK-Stop| 51682.73999999999|
|   Save-a-lot Markets|          40238.09|
|         Ernst Handel|          39975.91|
|       Mère Paillarde|22871.070000000003|
| Rattlesnake Canyo...|           17636.1|
|        Simons bistro|          16232.42|
| Hungry Owl All-Ni...|          14403.03|
|       Folk och fä HB|          13200.92|
|     HILARION-Abastos|          11799.74|
|   Berglunds snabbköp|          11758.92|
+---------------------+------------------+
only showing top 10 rows


# Q26 — CA par catégorie
Calculer le CA total par catégorie de produits. Afficher le nombre de produits distincts vendus par catégorie.